In [ ]:
!git clone -q https://github.com/shemshallah/nanlmv9.git
%cd nanlmv9
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
!mkdir -p data
!curl -sSL -o data/wt2_train.txt https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/train.txt
!python3 experiments/mamba_moe/tokenizer_bpe.py --vocab-texts data/wt2_train.txt --train-texts data/wt2_train.txt --out data/tok32k --vocab-size 32000

In [ ]:
!for p in aa ab ac ad ae af; do curl -L --retry 3 -o cm17.part-$p https://github.com/shemshallah/nanlmv9/releases/download/cm-prior-ctx17/cm17.part-$p; done
!cat cm17.part-* > cm_valid_full_ctx17.bin
!curl -L --retry 3 -o win_valid_full.bin https://github.com/shemshallah/nanlmv9/releases/download/cm-prior-ctx17/win_valid_full.bin
!ls -lh cm_valid_full_ctx17.bin win_valid_full.bin

In [ ]:
!python3 experiments/mamba_moe/train_word_char.py \
  --data-dir data/tok32k --cm-dump cm_valid_full_ctx17.bin --window win_valid_full.bin \
  --out runs/6h --steps 1000000 --tokens 64 \
  --d-model 512 --char-model 128 --experts 64 --lr 2e-4 \
  --scan chunked --scan-chunk 4 --device cuda --amp --max-seconds 21600 \
  --save-interval 1000 --result-json runs/6h/result.json

In [ ]:
import json; print(open('runs/6h/result.json').read())